# Transcribe a MultiWhispr recording

Takes the newest recording (or one you pick), runs Whisper on each channel separately, then merges both into one speaker-labelled transcript using the segment timing from `session.json`.

Runs fully locally with `openai-whisper`. Pick a model in the config cell; `turbo` is the best default on Apple Silicon.

In [ ]:
import json, subprocess, tempfile
from pathlib import Path
import numpy as np
from scipy.io import wavfile
import whisper
import warnings; warnings.filterwarnings("ignore")

MEETINGS = Path.home() / "Library/Application Support/MultiWhispr/Meetings"
MODEL = "turbo"          # tiny, base, small, medium, large-v3, turbo
LANGUAGE = None          # e.g. "en" or "ru"; None = auto-detect per channel
RECORDING = None         # folder name like "2026-10-26 12.44.03"; None = newest
MAX_SECONDS = None       # e.g. 300 to try the first five minutes only

folder = MEETINGS / RECORDING if RECORDING else sorted(p for p in MEETINGS.iterdir() if (p / "recording.wav").exists())[-1]
meta = json.loads((folder / "session.json").read_text())
rate, data = wavfile.read(folder / "recording.wav")
if MAX_SECONDS:
    data = data[: int(MAX_SECONDS * rate)]
print(folder.name, f"{len(data)/rate/60:.1f} min", {c["speaker"]: c["source"] for c in meta["channels"]})

## 1. Split channels

Left = `me`, right = `others`, exactly as `session.json` says. Each channel is written to its own mono file so Whisper never hears the other side.

In [ ]:
speakers = {c["speaker"]: c["index"] for c in meta["channels"]}
tmp = Path(tempfile.mkdtemp())
channel_files = {}
for speaker, index in speakers.items():
    mono = data[:, index]
    path = tmp / f"{speaker}.wav"
    wavfile.write(path, rate, mono)
    channel_files[speaker] = path
    active = float((np.abs(mono.astype(float) / 32768) > 0.004).mean())
    print(f"{speaker:7s} -> {path.name}  active {active:.0%}")

## 2. Transcribe each channel

Whisper is told where speech actually is (`clip_timestamps` from `session.json`) so it skips the long stretches where this speaker is silent. That is faster and avoids hallucinated text in silence.

In [ ]:
model = whisper.load_model(MODEL)

def clips_for(speaker, pad=0.3, min_gap=1.0):
    """Merge this speaker's segments into clip ranges Whisper should look at."""
    limit = len(data) / rate
    ranges = []
    for s in meta["segments"]:
        if s["speaker"] != speaker: continue
        start, end = max(0, s["start"] - pad), min(limit, s["end"] + pad)
        if start >= limit: continue
        if ranges and start - ranges[-1][1] < min_gap:
            ranges[-1][1] = max(ranges[-1][1], end)
        else:
            ranges.append([start, end])
    return [t for r in ranges for t in r]

# One conversation, one language: detect it on the channel with the most
# speech and apply it to both, otherwise a quiet side gets misdetected.
language = LANGUAGE
if language is None:
    busiest = max(channel_files, key=lambda s: sum(x["end"] - x["start"] for x in meta["segments"] if x["speaker"] == s))
    clips = clips_for(busiest)
    probe_start = clips[0] if clips else 0
    audio = whisper.load_audio(str(channel_files[busiest]))[int(probe_start * 16000): int((probe_start + 30) * 16000)]
    _, probs = model.detect_language(whisper.log_mel_spectrogram(whisper.pad_or_trim(audio), n_mels=model.dims.n_mels).to(model.device))
    language = max(probs, key=probs.get)
    print(f"detected language on '{busiest}': {language} ({probs[language]:.0%})")

results = {}
for speaker, path in channel_files.items():
    clips = clips_for(speaker)
    if not clips:
        print(f"{speaker}: no speech segments, skipped"); results[speaker] = []; continue
    out = model.transcribe(
        str(path),
        language=language,
        clip_timestamps=clips,
        condition_on_previous_text=False,
        word_timestamps=False,
        verbose=False,
    )
    segs = [s for s in out["segments"] if s["text"].strip() and s.get("no_speech_prob", 0) < 0.8]
    results[speaker] = segs
    print(f"{speaker}: {len(segs)} segments, language={out['language']}")

## 3. Merge into one dialogue

Segments from both speakers are interleaved by start time. Consecutive segments by the same speaker are joined.

In [ ]:
def fmt(t):
    m, s = divmod(int(t), 60); h, m = divmod(m, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

merged = sorted(
    ((s["start"], s["end"], speaker, s["text"].strip()) for speaker, segs in results.items() for s in segs),
    key=lambda x: x[0],
)
lines, current = [], None
for start, end, speaker, text in merged:
    if current and current["speaker"] == speaker and start - current["end"] < 2.0:
        current["text"] += " " + text; current["end"] = end
    else:
        if current: lines.append(current)
        current = {"start": start, "end": end, "speaker": speaker, "text": text}
if current: lines.append(current)

NAMES = {"me": "Me", "others": "Them"}
transcript = "\n".join(f"[{fmt(l['start'])}] {NAMES[l['speaker']]}: {l['text']}" for l in lines)
print(transcript[:3000])

## 4. Save next to the recording

Writes `transcript.txt` (readable) and `transcript.json` (per-line timing) into the recording folder. Nothing leaves the machine.

In [ ]:
(folder / "transcript.txt").write_text(transcript + "\n")
(folder / "transcript.json").write_text(json.dumps({"model": MODEL, "lines": lines}, ensure_ascii=False, indent=1))
print("saved to", folder)